# Classical optimizers — `algos/`

One `Optimizer` class family over three problem shapes (`algos/problems.py`):

- **`QUBOProblem`** — binary quadratic. `.from_microgrid(net, lambd)` wraps the microgrid-partitioning
  objective in `pp_to_microgrid.py`.
- **`MILPProblem`** — mixed-integer linear/quadratic.
- **`UnitCommitmentProblem`** — thermal unit commitment as a MILP + a per-generator DP for decomposition.

Optimizers: `brute_force`, `steepest_descent`, `simulated_annealing`, `monte_carlo` (parallel
tempering), `tabu`, `tree_decomposition` (exact), `gurobi` (MIQP/MILP), `highs` (scipy→HiGHS),
`lagrangian` (dual decomposition), `admm`. See [`docs/02-ALGORITHMS.md`](../docs/02-ALGORITHMS.md).

In [1]:
import os, sys, warnings
warnings.filterwarnings("ignore")
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".." if os.path.basename(os.getcwd()) == "notebooks" else "."))
if ROOT not in sys.path: sys.path.insert(0, ROOT)
os.chdir(ROOT)

import numpy as np, pandas as pd, pandapower as pp
import plotly.graph_objects as go, plotly.io as pio
from plotly.subplots import make_subplots
pio.renderers.default = "notebook_connected"
PAL = ["#DC267F", "#648FFF", "#FE6100", "#785EF0", "#FFB000", "#009E73", "#3DDBD9", "#808080"]

from algos import QUBOProblem, UnitCommitmentProblem, Optimizer, optimize, benchmark, plot_benchmark
print("optimizers:", sorted(Optimizer.registry()))

optimizers: ['admm', 'brute_force', 'gurobi', 'highs', 'lagrangian', 'monte_carlo', 'numpy_min_eigen', 'qaoa', 'random_search', 'simulated_annealing', 'steepest_descent', 'tabu', 'tree_decomposition', 'vqe']


## 1 · Microgrid partitioning as a QUBO

`pp_to_microgrid.microgrid_objective(net, lambd)` → symmetric objective; `to_QUBO` → binary
quadratic. `QUBOProblem.from_microgrid` packages both. `x_i ∈ {0,1}` is the partition group of bus `i`.

In [2]:
# a real converted network (GridSFM Rhode Island); fall back to the minimal example
try:
    from data.gridsfm_to_pp import GridSFMOut, gridsfm_to_pp
    g = GridSFMOut("rhode_island", "16h", "data/fixtures/gridsfm/rhode_island_model_16h.json"); g.read_data()
    net = gridsfm_to_pp(g.bus, g.gen, g.branch, g.load, g.shunt, g.dcline, g.baseMVA)
    net_name = "GridSFM Rhode Island"
except Exception:
    from pp_to_microgrid import create_minimal_example
    net = create_minimal_example(nbusses=4); net_name = "minimal example"

q = QUBOProblem.from_microgrid(net, lambd=0.5)
print(net_name, "->", q)
print("applies:", Optimizer.list(q))

df_q = benchmark(q)
df_q[["optimizer", "objective", "feasible", "runtime_ms", "opt_gap"]]

add_admittance_impedance ran in 0.007809s
min_sensitivity_matrix ran in 0.011966s
electrical_coupling_strength_matrix ran in 0.020483s
modularity_matrix ran in 0.020779s
self_reliance_matrix ran in 0.004729s
microgrid_objective ran in 0.025653s
GridSFM Rhode Island -> QUBOProblem('microgrid(lambd=0.5)', n=11)
applies: ['admm', 'brute_force', 'gurobi', 'highs', 'monte_carlo', 'numpy_min_eigen', 'qaoa', 'random_search', 'simulated_annealing', 'steepest_descent', 'tabu', 'tree_decomposition', 'vqe']


Restricted license - for non-production use only - expires 2027-11-29


,optimizer,objective,feasible,runtime_ms,opt_gap
0,monte_carlo,-0.247940,True,957.089500,-1.388114e-14
1,brute_force,-0.247940,True,6.832917,0.000000e+00
2,random_search,-0.247940,True,178.625292,0.000000e+00
3,steepest_descent,-0.247940,True,1.760458,0.000000e+00
4,highs,-0.247940,True,158.477250,0.000000e+00
5,qaoa,-0.247940,True,110629.936875,0.000000e+00
6,vqe,-0.247940,True,1566.712625,0.000000e+00
7,numpy_min_eigen,-0.247940,True,1.211167,0.000000e+00
8,tabu,-0.247940,True,211.739208,4.477787e-16
9,tree_decomposition,-0.247940,True,1.758708,4.477787e-16


In [3]:
plot_benchmark(df_q).show()

In [4]:
# the winning partition, drawn on the network
best = df_q.iloc[0]
res = optimize(q, best.optimizer)
part = res.x.astype(int)
print(f"{best.optimizer}: {part.sum()} buses in group 1, {len(part)-part.sum()} in group 0  (obj {res.objective:.4g})")

if "geo" in net.bus.columns and net.bus.geo.notna().any():
    import json as _json
    xy = net.bus.geo.map(lambda s: _json.loads(s)["coordinates"] if isinstance(s, str) else (None, None))
    lat = xy.map(lambda c: c[0]); lon = xy.map(lambda c: c[1])
    fig = go.Figure()
    for _, r in net.line.iterrows():
        fig.add_trace(go.Scatter(x=[lon[r.from_bus], lon[r.to_bus]], y=[lat[r.from_bus], lat[r.to_bus]],
                                 mode="lines", line=dict(color="#bbb", width=1), showlegend=False, hoverinfo="skip"))
    for grp, col in [(0, PAL[1]), (1, PAL[0])]:
        m = part == grp
        fig.add_trace(go.Scatter(x=lon[m], y=lat[m], mode="markers", name=f"group {grp}",
                                 marker=dict(size=11, color=col)))
    fig.update_layout(title=f"microgrid partition — {best.optimizer} ({net_name})",
                      xaxis_title="lon", yaxis_title="lat", height=380)
    fig.show()

monte_carlo: 7 buses in group 1, 4 in group 0  (obj -0.2479)


## 2 · Same microgrid QUBO, gate-model quantum (Qiskit)

`algos` also solves `QUBOProblem` with Qiskit — the same Ising Hamiltonian the D-Wave path anneals,
here minimised by a parameterised quantum circuit instead:

- **`qaoa`** — `QAOAAnsatz` (the textbook algorithm), angles trained by a classical optimizer
- **`vqe`** — a shallow hardware-efficient `RealAmplitudes` ansatz, same Hamiltonian
- **`numpy_min_eigen`** — exact ground state (dense diagonalisation) — the reference the other two are graded against

Both are simulated *exactly* (statevector, no shot noise) while optimizing, then the trained circuit
is sampled once to read out the best bitstring. `qaoa`'s cost circuit has one gate per nonzero QUBO
term, so its runtime scales with QUBO **density**, not just qubit count — a smaller/simpler instance
here (`nbusses=1`) keeps this cell fast; `docs/02-ALGORITHMS.md` has the perf/correctness notes from
getting this wired up (a naive Estimator call pattern was ~1000x slower; a `dimod` dict key-order bug
was silently dropping interaction terms).

In [5]:
from pp_to_microgrid import create_minimal_example

net_q = create_minimal_example(nbusses=1)
qq = QUBOProblem.from_microgrid(net_q, lambd=0.5)
print(qq, "n =", qq.n)

df_qk = benchmark(qq, names=["brute_force", "numpy_min_eigen", "vqe", "qaoa"],
                  qaoa={"reps": 1, "maxiter": 40})
df_qk[["optimizer", "objective", "feasible", "runtime_ms", "opt_gap"]]

add_admittance_impedance ran in 0.002625s
min_sensitivity_matrix ran in 0.005321s
electrical_coupling_strength_matrix ran in 0.008476s
modularity_matrix ran in 0.008739s
self_reliance_matrix ran in 0.003478s
microgrid_objective ran in 0.012373s
QUBOProblem('microgrid(lambd=0.5)', n=7) n = 7


,optimizer,objective,feasible,runtime_ms,opt_gap
0,brute_force,0.250311,True,0.416791,0.0
1,numpy_min_eigen,0.250311,True,0.801125,0.0
2,vqe,0.250311,True,896.033125,0.0
3,qaoa,0.250311,True,343.035875,0.0


In [6]:
fig = make_subplots(rows=1, cols=2, subplot_titles=("objective (all exact)", "QAOA convergence (best so far)"))
fig.add_trace(go.Bar(x=df_qk.optimizer, y=df_qk.objective, marker_color=PAL[3], showlegend=False), row=1, col=1)
qaoa_trace = df_qk.attrs["traces"].get("qaoa", [])
fig.add_trace(go.Scatter(y=qaoa_trace, mode="lines+markers", line_color=PAL[0], showlegend=False), row=1, col=2)
fig.update_xaxes(title_text="COBYLA iteration", row=1, col=2)
fig.update_layout(height=320, title="microgrid QUBO -- classical exact vs Qiskit QAOA/VQE")
fig.show()

## 3 · Unit commitment

`min Σ (marginal·p + startup·v)` s.t. per-period demand balance (the **coupling** constraint),
`pmin·u ≤ p ≤ pmax·u`, and min-up/min-down times. `gurobi` / `highs` solve the monolithic MILP;
`lagrangian` relaxes the demand balance into one single-unit DP per generator; `admm` splits across
generators.

In [7]:
uc = UnitCommitmentProblem.example(T=24, seed=1)
print(uc, "— peak demand", uc.demand.max().round(1), "MW,  total capacity",
      sum(g.pmax for g in uc.gens), "MW")

df_uc = benchmark(uc)
df_uc[["optimizer", "objective", "feasible", "runtime_ms", "bound", "opt_gap"]]

UnitCommitmentProblem('uc_example(T=24)', G=5, T=24) — peak demand 567.0 MW,  total capacity 1130 MW


,optimizer,objective,feasible,runtime_ms,bound,opt_gap
0,gurobi,187849.202178,True,23.903500,187849.202178,0.000000e+00
1,highs,187849.202178,True,110.565250,187849.202178,1.239455e-15
2,lagrangian,187849.202178,True,263.568291,183838.596811,1.239455e-15
3,admm,211269.389055,True,221.572000,NaN,1.246755e-01


In [8]:
plot_benchmark(df_uc).show()

In [9]:
# dispatch stack for the best schedule + Lagrangian dual-bound convergence
best_uc = df_uc.iloc[0]
res_uc = optimize(uc, best_uc.optimizer)
P = res_uc.x.reshape(uc.G, uc.T)

fig = make_subplots(rows=1, cols=2, column_widths=[0.6, 0.4],
                    subplot_titles=(f"dispatch stack — {best_uc.optimizer} (${res_uc.objective:,.0f})",
                                    "Lagrangian: primal cost vs dual bound"))
cum = np.zeros(uc.T)
for gi in range(uc.G):
    fig.add_trace(go.Scatter(x=list(range(uc.T)), y=cum + P[gi], fill="tonexty" if gi else "tozeroy",
                             mode="lines", name=f"gen {gi} (${uc.gens[gi].cost}/MWh)",
                             line=dict(width=0.5, color=PAL[gi % len(PAL)])), row=1, col=1)
    cum = cum + P[gi]
fig.add_trace(go.Scatter(x=list(range(uc.T)), y=uc.demand, mode="lines+markers", name="demand",
                         line=dict(color="black", dash="dash")), row=1, col=1)

lag = optimize(uc, "lagrangian", iters=150)
fig.add_trace(go.Scatter(y=lag.trace, mode="lines", name="best primal", line_color=PAL[0]), row=1, col=2)
fig.add_hline(y=lag.bound, line_dash="dot", line_color=PAL[1], row=1, col=2,
              annotation_text=f"dual bound {lag.bound:,.0f}")
fig.update_xaxes(title_text="hour", row=1, col=1); fig.update_xaxes(title_text="iteration", row=1, col=2)
fig.update_layout(height=380, legend=dict(orientation="h", y=-0.25))
fig.show()
print(f"lagrangian duality gap: {lag.gap:.2%}   (primal ${lag.objective:,.0f}, dual bound ${lag.bound:,.0f})")

lagrangian duality gap: 7.31%   (primal $187,849, dual bound $174,109)


## 4 · Which optimizer when

- **exact, small**: `brute_force` (QUBO n≤22), `tree_decomposition` (low treewidth), `gurobi` / `highs` (MILP)
- **fast heuristic**: `steepest_descent` (surprisingly strong), then `simulated_annealing` / `tabu`
- **rugged landscape**: `monte_carlo` (parallel tempering)
- **structured / decomposable**: `lagrangian` (gives a bound + gap), `admm` (consensus split, warm-startable)
- **quantum-inspired baseline to beat**: `dwave-samplers` path (`simulated_annealing(backend="dwave")`, `tabu`)
- **gate-model quantum**: `vqe` (fast, generic ansatz) to sanity-check the QUBO is solvable at all;
  `qaoa` when the algorithm itself is what you're evaluating — budget for its cost scaling with density

In [10]:
summary = pd.concat([df_q.assign(problem="microgrid QUBO"), df_qk.assign(problem="microgrid QUBO (Qiskit)"),
                     df_uc.assign(problem="unit commitment")],
                    ignore_index=True)[["problem", "optimizer", "objective", "feasible", "runtime_ms", "opt_gap"]]
summary.round({"objective": 4, "runtime_ms": 1, "opt_gap": 4})

,problem,optimizer,objective,feasible,runtime_ms,opt_gap
0,microgrid QUBO,monte_carlo,-0.2479,True,957.1,-0.0000
1,microgrid QUBO,brute_force,-0.2479,True,6.8,0.0000
2,microgrid QUBO,random_search,-0.2479,True,178.6,0.0000
3,microgrid QUBO,steepest_descent,-0.2479,True,1.8,0.0000
4,microgrid QUBO,highs,-0.2479,True,158.5,0.0000
...,...,...,...,...,...,...
16,microgrid QUBO (Qiskit),qaoa,0.2503,True,343.0,0.0000
17,unit commitment,gurobi,187849.2022,True,23.9,0.0000
18,unit commitment,highs,187849.2022,True,110.6,0.0000
19,unit commitment,lagrangian,187849.2022,True,263.6,0.0000
